In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path("/kaggle/working/SEEP_N1")
BRANCH = "feat/n1-spectral-response-memory"
if not REPO.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH,
                    "https://github.com/thanhphu25/SEEP.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
ENV = Path("/kaggle/working/seep_n1_env")
PY = str(ENV / "bin/python")
if not Path(PY).exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.10",
                    "--seed", str(ENV)], check=True)
subprocess.run([PY, "-m", "pip", "install", "torch==2.0.1", "torchvision==0.15.2",
                "--index-url", "https://download.pytorch.org/whl/cu118"], check=True)
subprocess.run([PY, "-m", "pip", "install", "numpy==1.26.4", "Pillow==10.3.0",
                "timm==0.6.11", "loguru==0.7.2", "huggingface-hub==0.26.2"], check=True)
subprocess.run([PY, "-c", "import torch, torchvision, timm, main; "
                "print('torch', torch.__version__, 'torchvision', torchvision.__version__, 'timm', timm.__version__); "
                "assert torch.cuda.is_available(), 'Enable GPU in Kaggle notebook settings'; "
                "print('GPU:', torch.cuda.get_device_name(0))"], check=True)


In [ ]:
DATA_ROOT = Path("/kaggle/input/YOUR_CUB_DATASET")  # EDIT THIS PATH
CONFIG = "cub200_10task"  # or "cub200_20task"
SEED = 3407
MODE, LAMBDA = "semantic_full", 1.0

if DATA_ROOT.name == "CUB_200_2011":
    DATA_ROOT = DATA_ROOT.parent
cub = DATA_ROOT / "CUB_200_2011"
if not all((cub / split).is_dir() for split in ("train", "test")):
    raw = cub if (cub / "images.txt").is_file() else DATA_ROOT
    assert (raw / "images.txt").is_file(), f"Set DATA_ROOT to your CUB dataset: {DATA_ROOT}"
    prepared = Path("/kaggle/working/cub_n1_prepared")
    subprocess.run([PY, "scripts/prepare_cub.py", "--source", str(raw),
                    "--output-root", str(prepared), "--method", "symlink"], check=True)
    DATA_ROOT = prepared
OUT = Path(f"/kaggle/working/{CONFIG}_n1_{MODE}_lambda{LAMBDA}_seed{SEED}")
CMD = [PY, "scripts/run_cub_n1.py", "--config", CONFIG, "--seed", str(SEED),
       "--data-path", str(DATA_ROOT), "--output-dir", str(OUT), "--srm-mode", MODE,
       "--srm-lambda", str(LAMBDA), "--srm-proj-dim", "16", "--srm-samples-per-class", "8",
       "--srm-probe-batch-size", "1", "--srm-memory-seed", "3407", "--srm-normalize", "trace"]
print("Output:", OUT)


In [ ]:
import json
subprocess.run(CMD + ["--dry-run"], check=True)
report = json.loads((OUT / "effective_config.json").read_text())
print("Config:", report["config_source"], report["args"]["subparser_name"])
print("Learning rates:", report["learning_rates"])
print("N1:", {k: v for k, v in report["args"].items() if k.startswith("srm_")})


In [ ]:
subprocess.run(CMD, check=True)
